In [4]:
#imports for Code to run
import numpy as np
import pandas as pd#Data commands
from google.colab import files#allows file transfers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.tree import DecisionTreeClassifier, export_text

sensor_Readings = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]#Table names


WINDOW_SIZE = 50 # HOW MUCH DATA WILL BE USED IN A SAMPLE

STEP = 25 #SAMPLE SPLIT


uploaded = files.upload()          # pick both CSV files
file_names = list(uploaded.keys())#uploaded.keys pulls the file names for the "list()" command to put them in a list 
dfs = [pd.read_csv(name) for name in file_names]#Iterates using a for loop by reading the files using the pandas command .read_csv then this command will create A dataframe(which is a code version of the database) of each csv file to use for the ML model to read
#In short makes csv file into easily machine readable code

#Prints the lines to check for how many there are in the dataframes
print("Loaded:", file_names)
for d in dfs:
    print(d.shape, list(d.columns))



ModuleNotFoundError: No module named 'numpy'

In [ ]:
#Clarifies what is a sitting and what is walking
for name, d in zip(file_names, dfs): # checks the file names and dataframes "Dataframes" = copy of data in files
    if "sitting" in name.lower(): # reads the name of the file
        d["label"] = 0 # holds file dataframe and adds a label column
    elif "standing" in name.lower(): # reads the name of the file
        d["label"] = 1 # holds file dataframe and adds a label column
    else:
        raise ValueError(f"Can't tell the label from filename: {name}")


In [ ]:
##------------------------------This section preps the data for its use inn the decison tree by picking 50 columns at a time and going through the whole csv file then getting min max mean and standard deviation------------------##

def make_windows(df):#DEFINES a function called make_windows
    # Extract only the sensor readings (the first 6 columns of the DataFrame) by position
    values = df.iloc[:, :6].values#converts first 6 columns into a numPy array ":" means select all rows and the ":6" means 6 columns. TEMPLATE FOR RECREATION = .iloc[ROW, COLUMN]
    labels = df["label"].values#copies the label if statements above to add the values of 1 or 0 to the numpPy array to correct columns
    X, y = [], []#creates empty lists for things to be added
    for start in range(0, len(df) - WINDOW_SIZE + 1, STEP):#uses the values in WINDOW_SIZE and STEP to make a loop that will find 25 window of data(WINDOW = 25 columns of data ) and will condense them into the mean of the 25 numbers in the for loop
        end = start + WINDOW_SIZE# marks where the end of a window is so another can be started
        win = values[start:end]#copies the values for the current sensor readings and adds them to array called win
        # Use the most common label in the window
        lab = np.bincount(labels[start:end]).argmax()#takes values from labels in the Windows created so from 50 values using "np.bincount"it checks and counts how many 1s and how moany 0s there are and returns them in an array in this case there are only 1 or 0 per file so the window will return an array of [0,50]
        # ".argmax" will return the the result of the number returned the most by the array so if there are 36 1s and 2 0s then it will return 1
        feats = np.concatenate([#the name is Feat short for features which means an input value a model uses to make a decison // ".concatenate" = is used to join data into one array
            ##| | |
            ##v v v uses the win array and numPy commands to get 4 different measurments of each column so they can be put into this array essentially summarising the data from the tables to get 6 numbers 1 for each measurment value
           #"axis=0" means tally up all numbers in vertical direction and "axis=1" tallys the numbers horizontally and it does this for all the rows avaialble so it gets 6 numbers one for each axis on the gyroscope and accerlerometer
           win.mean(axis=0),#Gets the mean of the WINDOW_SIZE so basically the mean of 50 numbers
            win.std(axis=0),#Finds standard deviation of the 50 numbers
            win.min(axis=0),#finds smallest value in the 50 numbers
            win.max(axis=0),#finds largest value in the 50 numbers
        ])
        X.append(feats)#Adds to the empty list the results of the 6 numbers
        y.append(lab)#Adds to the empy list created earlier the results ]of the argmax
    return np.array(X), np.array(y)#puts all of the data into array from the lists

# Window each file separately so no window spans two files
all_X, all_y = [], []#creates another list
for d in dfs:#Loops though dfs
    Xi, yi = make_windows(d)#Make windows function froom above is called in xi and yi
    all_X.append(Xi)#makes copy of the mean, std,min and max X is the numbers the model usess
    all_y.append(yi)#makes copy of 1 and 0s Y is the verification of what the numbers are

X = np.vstack(all_X)#Creates 2D array to stack the the current results into one table but under the already assigned data
y = np.concatenate(all_y)#joins the lists to make a
print("Feature matrix:", X.shape, "| label counts:", np.bincount(y))#"np.wsbincount"it checks and counts how many 1s and how moany 0s there are and returns them in an array in this case there are only 1 or 0 per file so the window will return an array of [0,50]

Feature matrix: (1090, 24) | label counts: [967 123]


In [ ]:
##---------------------MAKES DECISON TREE and gets the model to use it to predict answers---------------------##


#STRATIFY = y will ensure the test data it is trained on is equal for sitting and standing usign the bool classifications in the label catagory
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y)#Creates the test split with the "test_size"with the program using the percentage in this case 0.2 for test data and the rest /80 is the training set and it helps the model learn and make predictions based on this data to guess the correct label for the 20% of the data it will try to guess after it is trained
model = DecisionTreeClassifier(max_depth=4, random_state=42)#random state is the answer to the ultimate question of life, but its purpose is just to shuffle the data the 42 is like a seed for how to shuffle (Like minecraft the shufffle is random but repeatable with the same seed)
#// MAX_depth gives the model a set amount of questions to allow it to identify if somthing is sitting or standing naturally this will likely make a model more accurate if it can ask more questions about the data
model.fit(X_train, y_train)#Learns using the questions from X_train and uses Y_train to get the estimates made from the questions to be used in the model variable usign ".fit"

y_pred = model.predict(X_test)#tests the model by letting it train on the test again
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))#marks how accurate it was with "accuracy_score()"" the number at the end controlling the decimal place of the accuracy score this case being 3 decimal places
print(classification_report(y_test, y_pred, target_names=["sitting", "walking"]))#gives a induvidual accuracy score for both the values so you can see if it was better at guessing sitting or standing. using "classification_report()" "target_names" gives visual names to the output

# Print the tree as readable rules
# Features are ordered as all means, then all stds, then all mins, then all maxes
feature_names = [f"{s}_{stat}" for stat in ["mean", "std", "min", "max"] for s in sensor_Readings]#class 1 = walking/ class 0 =sitting
print(export_text(model, feature_names))#shows the decsion tree model and how it decided if somthing was sittign or standing with this if it was more than or equal to -0.22 then it would be class one
#DOES DECISION TREE

Accuracy: 1.0
              precision    recall  f1-score   support

     sitting       1.00      1.00      1.00       193
    standing       1.00      1.00      1.00        25

    accuracy                           1.00       218
   macro avg       1.00      1.00      1.00       218
weighted avg       1.00      1.00      1.00       218

|--- gyro_x_min <= -0.22
|   |--- class: 1
|--- gyro_x_min >  -0.22
|   |--- class: 0

